In [ ]:
"""
OpenAudit: Reproducible Analysis Pipeline
==========================================

Purpose
-------
Reproduce the quantitative transparency analyses reported in the OpenAudit
study from the publicly released model/version-level audit dataset.

Required packages
-----------------
pandas
numpy
scipy
openpyxl

Input
-----
A model/version-level Excel or CSV file containing verified audit fields.

IMPORTANT
---------
Column names below are illustrative. Rename them to exactly match the
publicly released OpenAudit dataset.
"""

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_FILE = Path("data/OpenAudit_dataset.xlsx")
SHEET_NAME = "Audit Data"
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_BOOTSTRAP = 10_000


# ============================================================
# 2. DEFINE OPENAUDIT VARIABLES
# ============================================================

# Seven binary disclosure components used in CTS.
CTS_COMPONENTS = [
    "source_code",
    "model_weights",
    "training_data",
    "preprocessing",
    "evaluation_protocol",
    "technical_documentation",
    "alignment_safety",
]

# Five training-data disclosure dimensions used in TDDI.
TDDI_COMPONENTS = [
    "data_source_specification",
    "preprocessing_methodology",
    "licensing_clarity",
    "language_domain_distribution",
    "synthetic_organic_composition",
]

# Four artifact-availability dimensions used in RS.
RS_COMPONENTS = [
    "source_code",
    "model_weights",
    "training_data",
    "evaluation_protocol",
]


# ============================================================
# 3. LOAD DATA
# ============================================================

def load_data(path, sheet_name=None):
    """Load the released OpenAudit dataset."""

    if path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path, sheet_name=sheet_name)

    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)

    else:
        raise ValueError("Input must be an Excel or CSV file.")

    return df


df = load_data(DATA_FILE, SHEET_NAME)

print(f"Loaded {len(df)} model/version records.")


# ============================================================
# 4. VALIDATE REQUIRED VARIABLES
# ============================================================

def validate_columns(df, required_columns):
    """Verify that required audit variables exist."""

    missing = [c for c in required_columns if c not in df.columns]

    if missing:
        raise ValueError(
            "Required columns are missing:\n" +
            "\n".join(missing)
        )


validate_columns(
    df,
    list(set(CTS_COMPONENTS + TDDI_COMPONENTS + RS_COMPONENTS))
)


# ============================================================
# 5. VALIDATE BINARY DISCLOSURE SCORES
# ============================================================

def validate_binary_values(df, columns):
    """
    Verify that scored disclosure variables contain only
    0, 1, or missing values.

    Missing/ambiguous/inaccessible evidence must be resolved according
    to the published OpenAudit protocol before metric computation.
    """

    for column in columns:

        observed = df[column].dropna()

        invalid = observed[~observed.isin([0, 1])]

        if len(invalid) > 0:
            raise ValueError(
                f"{column} contains values other than 0/1."
            )


validate_binary_values(
    df,
    list(set(CTS_COMPONENTS + TDDI_COMPONENTS + RS_COMPONENTS))
)


# ============================================================
# 6. COMPOSITE TRANSPARENCY SCORE (CTS)
# ============================================================

def calculate_cts(row):
    """
    Composite Transparency Score.

              1   7
    CTS_m =  ---  Σ d_mi
              7  i=1

    where d_mi is the verified binary disclosure indicator.
    """

    values = row[CTS_COMPONENTS]

    # Do not silently treat missing evidence as zero.
    if values.isna().any():
        return np.nan

    return values.mean()


df["CTS"] = df.apply(calculate_cts, axis=1)


# ============================================================
# 7. TRAINING DATA DISCLOSURE INDEX (TDDI)
# ============================================================

def calculate_tddi(row):
    """
    Training Data Disclosure Index.

               1   5
    TDDI_m =  ---  Σ b_mj
               5  j=1
    """

    values = row[TDDI_COMPONENTS]

    if values.isna().any():
        return np.nan

    return values.mean()


df["TDDI"] = df.apply(calculate_tddi, axis=1)


# ============================================================
# 8. DEPTH DEFICIENCY INDEX (DDI)
# ============================================================

def calculate_ddi(row):
    """
    Seven-dimensional Depth Deficiency Index.

              1   7
    DDI_m =  ---  Σ (1 - d_mi)
              7  i=1

    Higher values indicate greater disclosure deficiency.
    """

    values = row[CTS_COMPONENTS]

    if values.isna().any():
        return np.nan

    return (1 - values).mean()


df["DDI"] = df.apply(calculate_ddi, axis=1)


# Mathematical consistency check.
valid = df[["CTS", "DDI"]].dropna()

assert np.allclose(
    valid["CTS"] + valid["DDI"],
    1.0,
    atol=1e-10
)


# ============================================================
# 9. REPRODUCIBILITY SCORE (RS)
# ============================================================

def calculate_rs(row):
    """
    Reproducibility-oriented artifact availability score.

          a_code + a_weights + a_data + a_eval
    RS = ---------------------------------------
                         4
    """

    values = row[RS_COMPONENTS]

    if values.isna().any():
        return np.nan

    return values.mean()


df["RS"] = df.apply(calculate_rs, axis=1)


# ============================================================
# 10. FIVE-FIELD TRAINING-DATA DISCLOSURE DEPTH
# ============================================================

def training_data_depth(row):
    """
    Compute the mean disclosure depth across the five
    training-data dimensions.

    This measure is analytically distinct from the formal DDI.
    """

    values = row[TDDI_COMPONENTS]

    if values.isna().any():
        return np.nan

    return values.mean()


df["Training_Data_Depth"] = df.apply(
    training_data_depth,
    axis=1
)

df["Training_Data_Incompleteness"] = (
    1 - df["Training_Data_Depth"]
)


# ============================================================
# 11. TRANSPARENCY BANDS
# ============================================================

def transparency_band(score):
    """
    Classify CTS into transparency bands used by the study.
    """

    if pd.isna(score):
        return np.nan

    if score <= 0.25:
        return "Low"

    elif score >= 0.75:
        return "High"

    else:
        return "Intermediate"


df["CTS_Band"] = df["CTS"].apply(transparency_band)


# ============================================================
# 12. DESCRIPTIVE STATISTICS
# ============================================================

def descriptive_statistics(series):
    """Return core descriptive statistics."""

    x = series.dropna()

    return {
        "N": len(x),
        "Mean": x.mean(),
        "Median": x.median(),
        "SD": x.std(ddof=1),
        "Minimum": x.min(),
        "Maximum": x.max(),
    }


metrics = [
    "CTS",
    "TDDI",
    "DDI",
    "RS",
    "Training_Data_Depth",
    "Training_Data_Incompleteness",
]

summary = pd.DataFrame({
    metric: descriptive_statistics(df[metric])
    for metric in metrics
}).T

print("\nOverall descriptive statistics")
print(summary)


# ============================================================
# 13. BOOTSTRAP 95% CONFIDENCE INTERVAL
# ============================================================

def bootstrap_mean_ci(
    series,
    n_bootstrap=10_000,
    confidence=0.95,
    seed=42
):
    """
    Nonparametric bootstrap confidence interval for the mean.
    """

    x = series.dropna().to_numpy()

    rng = np.random.default_rng(seed)

    bootstrap_means = np.empty(n_bootstrap)

    for i in range(n_bootstrap):

        sample = rng.choice(
            x,
            size=len(x),
            replace=True
        )

        bootstrap_means[i] = sample.mean()

    alpha = 1 - confidence

    lower = np.quantile(
        bootstrap_means,
        alpha / 2
    )

    upper = np.quantile(
        bootstrap_means,
        1 - alpha / 2
    )

    return x.mean(), lower, upper


cts_mean, cts_low, cts_high = bootstrap_mean_ci(
    df["CTS"],
    N_BOOTSTRAP,
    seed=RANDOM_SEED
)

rs_mean, rs_low, rs_high = bootstrap_mean_ci(
    df["RS"],
    N_BOOTSTRAP,
    seed=RANDOM_SEED
)

print(
    f"\nCTS mean = {cts_mean:.4f}; "
    f"95% bootstrap CI = [{cts_low:.4f}, {cts_high:.4f}]"
)

print(
    f"RS mean = {rs_mean:.4f}; "
    f"95% bootstrap CI = [{rs_low:.4f}, {rs_high:.4f}]"
)


# ============================================================
# 14. TRANSPARENCY-BAND DISTRIBUTION
# ============================================================

band_counts = (
    df["CTS_Band"]
    .value_counts()
    .rename_axis("Transparency_Band")
    .reset_index(name="Count")
)

band_counts["Percentage"] = (
    100 * band_counts["Count"] / df["CTS"].notna().sum()
)

print("\nCTS transparency bands")
print(band_counts)


# ============================================================
# 15. DISCLOSURE-COMPONENT PREVALENCE
# ============================================================

component_prevalence = []

for component in CTS_COMPONENTS:

    valid_component = df[component].dropna()

    component_prevalence.append({
        "Component": component,
        "Disclosed_N": int(valid_component.sum()),
        "Evaluated_N": len(valid_component),
        "Percentage": 100 * valid_component.mean(),
    })


component_prevalence = pd.DataFrame(
    component_prevalence
).sort_values(
    "Disclosed_N",
    ascending=False
)

print("\nDisclosure-component prevalence")
print(component_prevalence)


# ============================================================
# 16. MODEL-CATEGORY ANALYSIS
# ============================================================

def category_summary(df):
    """
    Compare transparency metrics across model-access categories.

    Expected examples:
    open-source
    open-weight
    proprietary

    Category labels must come directly from the released audit data.
    """

    return (
        df.groupby("category")
        .agg(
            N=("CTS", "count"),
            CTS_Mean=("CTS", "mean"),
            CTS_SD=("CTS", "std"),
            TDDI_Mean=("TDDI", "mean"),
            DDI_Mean=("DDI", "mean"),
            RS_Mean=("RS", "mean"),
        )
        .reset_index()
    )


category_results = category_summary(df)

print("\nCategory-level results")
print(category_results)


# ============================================================
# 17. TEMPORAL ANALYSIS
# ============================================================

year_results = (
    df.groupby("release_year")
    .agg(
        N=("CTS", "count"),
        CTS_Mean=("CTS", "mean"),
        CTS_SD=("CTS", "std"),
        TDDI_Mean=("TDDI", "mean"),
        DDI_Mean=("DDI", "mean"),
        RS_Mean=("RS", "mean"),
    )
    .reset_index()
    .sort_values("release_year")
)

print("\nYear-level transparency")
print(year_results)


# ============================================================
# 18. PEARSON AND SPEARMAN ASSOCIATIONS
# ============================================================

def correlation_analysis(df, x, y):

    temp = df[[x, y]].dropna()

    pearson_r, pearson_p = stats.pearsonr(
        temp[x],
        temp[y]
    )

    spearman_rho, spearman_p = stats.spearmanr(
        temp[x],
        temp[y]
    )

    return {
        "Variables": f"{x} vs {y}",
        "N": len(temp),
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "Spearman_rho": spearman_rho,
        "Spearman_p": spearman_p,
    }


# Example: relationship between transparency and deficiency.
correlation_results = correlation_analysis(
    df,
    "CTS",
    "DDI"
)

print("\nCorrelation analysis")
print(correlation_results)


# ============================================================
# 19. KRUSKAL-WALLIS TEST
# ============================================================

def kruskal_by_category(df, metric, category="category"):
    """
    Nonparametric comparison of metric distributions
    across model-access categories.
    """

    groups = []

    labels = []

    for name, group in df.groupby(category):

        values = group[metric].dropna().to_numpy()

        if len(values) > 0:
            groups.append(values)
            labels.append(name)

    if len(groups) < 2:
        raise ValueError(
            "At least two groups are required."
        )

    H, p = stats.kruskal(*groups)

    return {
        "Metric": metric,
        "Groups": ", ".join(labels),
        "H": H,
        "p_value": p,
    }


kw_cts = kruskal_by_category(
    df,
    "CTS"
)

kw_rs = kruskal_by_category(
    df,
    "RS"
)

print("\nKruskal-Wallis: CTS")
print(kw_cts)

print("\nKruskal-Wallis: RS")
print(kw_rs)


# ============================================================
# 20. RS EXTREME VALUES
# ============================================================

rs_complete = (df["RS"] == 1).sum()
rs_zero = (df["RS"] == 0).sum()

print(f"\nModels with RS = 1: {rs_complete}")
print(f"Models with RS = 0: {rs_zero}")


# ============================================================
# 21. ENVIRONMENTAL REPORTING CALCULATION
# ============================================================

def operational_emissions_tco2e(
    electricity_kwh,
    carbon_intensity_kg_per_kwh
):
    """
    Estimate operational emissions.

    E_CO2e = E * I / 1000

    electricity_kwh:
        Electricity consumption in kWh.

    carbon_intensity_kg_per_kwh:
        Carbon intensity in kg CO2e/kWh.

    Returns
    -------
    Metric tonnes CO2e.

    This calculation represents operational electricity-related
    emissions only and does not estimate full life-cycle emissions.
    """

    return (
        electricity_kwh
        * carbon_intensity_kg_per_kwh
        / 1000
    )


# ============================================================
# 22. DATA-QUALITY / PROVENANCE CHECKS
# ============================================================

def provenance_audit(df):
    """
    Check whether core provenance fields required by the
    OpenAudit protocol are populated.

    Rename these variables to match the released dataset.
    """

    provenance_columns = [
        "model_name",
        "model_version",
        "evidence_source",
        "source_date",
        "access_date",
        "evidence_location",
        "scoring_rule",
        "decision_rationale",
    ]

    existing = [
        c for c in provenance_columns
        if c in df.columns
    ]

    if not existing:
        print(
            "No provenance columns found under the configured names."
        )
        return None

    report = pd.DataFrame({
        "Field": existing,
        "Available_N": [
            df[c].notna().sum()
            for c in existing
        ],
        "Missing_N": [
            df[c].isna().sum()
            for c in existing
        ],
    })

    return report


provenance_report = provenance_audit(df)

if provenance_report is not None:
    print("\nEvidence-provenance completeness")
    print(provenance_report)


# ============================================================
# 23. INTERNAL CONSISTENCY CHECKS
# ============================================================

def run_consistency_checks(df):

    for metric in ["CTS", "TDDI", "DDI", "RS"]:

        values = df[metric].dropna()

        assert values.between(0, 1).all(), (
            f"{metric} contains values outside [0,1]."
        )

    # CTS and DDI are complementary under the formal definitions.
    subset = df[["CTS", "DDI"]].dropna()

    assert np.allclose(
        subset["CTS"] + subset["DDI"],
        1.0,
        atol=1e-10
    )

    print(
        "\nAll configured metric consistency checks passed."
    )


run_consistency_checks(df)


# ============================================================
# 24. EXPORT REPRODUCIBLE RESULTS
# ============================================================

summary.to_csv(
    OUTPUT_DIR / "descriptive_statistics.csv"
)

band_counts.to_csv(
    OUTPUT_DIR / "cts_transparency_bands.csv",
    index=False
)

component_prevalence.to_csv(
    OUTPUT_DIR / "component_prevalence.csv",
    index=False
)

category_results.to_csv(
    OUTPUT_DIR / "category_results.csv",
    index=False
)

year_results.to_csv(
    OUTPUT_DIR / "year_results.csv",
    index=False
)

if provenance_report is not None:
    provenance_report.to_csv(
        OUTPUT_DIR / "provenance_completeness.csv",
        index=False
    )


# Save model-level calculated metrics.
output_columns = [
    c for c in [
        "model_name",
        "model_version",
        "developer",
        "category",
        "release_year",
        "CTS",
        "TDDI",
        "DDI",
        "RS",
        "Training_Data_Depth",
        "Training_Data_Incompleteness",
        "CTS_Band",
    ]
    if c in df.columns
]

df[output_columns].to_csv(
    OUTPUT_DIR / "model_level_metrics.csv",
    index=False
)

print("\nOpenAudit analysis completed successfully.")